# 🧬 ESM3 Tools Example

This notebook demonstrates how to use **ESM3** for protein embeddings, mutation sampling, sequence scoring, and structure prediction.

## 📖 What is ESM3?

[ESM3](https://www.science.org/doi/10.1126/science.adl2528) is a next-generation multimodal protein language model that jointly reasons over sequence, structure, and function.

### Key Features:
- **Embeddings** -- Rich vector representations capturing sequence, structure, and function
- **Sampling** -- Generate mutated sequence variants using model-guided proposals
- **Scoring** -- Pseudo-perplexity via masked scoring to assess sequence naturalness
- **Structure prediction** -- Generate 3D PDB structures with per-residue confidence metrics
- **Multimodal** -- Jointly models sequence, structure, and function tokens

## 📥 Imports

## 📦 Shared Data Types

### `SequenceScores`
Scoring metrics for a single sequence, returned by the scoring tool. Metrics can be accessed via dict-style (`score.metrics["perplexity"]`) or attribute-style (`score.perplexity`).

| Field | Type | Description |
|-------|------|-------------|
| `metrics` | `Dict[str, float]` | Scoring metrics: `log_likelihood`, `avg_log_likelihood`, `perplexity` |
| `logits` | `Optional[List[List[float]]]` | Per-position logits (seq_len, vocab_size). Only present if `return_logits=True` |
| `vocab` | `Optional[List[str]]` | Token ordering for logits columns |

In [ ]:
from bio_programming_tools.tools.masked_models.esm3 import (
    ESM3EmbeddingsInput, ESM3EmbeddingsConfig, run_esm3_embeddings,
    ESM3SampleInput, ESM3SampleConfig, run_esm3_sample,
    ESM3ScoringInput, ESM3ScoringConfig, run_esm3_score,
    ESM3StructurePredictionInput, ESM3StructurePredictionConfig, run_esm3_structure_prediction,
)

## 🔍 1. Extract Embeddings

Compute mean-pooled embeddings for protein sequences. ESM3 generates rich vector representations that capture structural, functional, and evolutionary properties of proteins. These embeddings can be used for downstream tasks like clustering, classification, and similarity search.

### API Reference

**`ESM3EmbeddingsInput`**
| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Protein sequence(s) to embed. Accepts a single string or list of strings |

**`ESM3EmbeddingsConfig`**
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_checkpoint` | `str` | `"esm3_sm_open_v1"` | ESM3 model checkpoint to use |
| `batch_size` | `int` | `128` | Number of sequences per batch |
| `return_logits` | `bool` | `False` | Whether to include per-position logits in the output |

**`ESM3EmbeddingsOutput`**
| Field | Type | Description |
|-------|------|-------------|
| `mean_embeddings` | `Optional[List[List[float]]]` | Mean-pooled embeddings (num_sequences, embedding_dim) |
| `num_sequences` | `int` | Number of sequences processed |
| `logits` | `Optional[List[List[List[float]]]]` | Per-position amino acid logits (num_sequences, seq_len, vocab_size) |
| `attention_masks` | `List[List[int]]` | Attention masks (num_sequences, seq_len) |

In [2]:
sequences = [
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
    "MNLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
]
inputs = ESM3EmbeddingsInput(sequences=sequences)

# Configure the embedding model
config = ESM3EmbeddingsConfig(
    model_checkpoint="esm3_sm_open_v1",
    batch_size=2,
    return_logits=False,
)

embeddings_result = run_esm3_embeddings(inputs, config)
print(f"Processed {embeddings_result.num_sequences} sequences")
print(f"Embedding dim: {len(embeddings_result.mean_embeddings[0])}")


Processed 2 sequences
Embedding dim: 1536


## 🧪 2. Sample Mutations

Generate mutated sequence variants using model-guided proposals. ESM3 uses its learned amino acid distributions to propose mutations at positions where the model is most uncertain or most confident, depending on the decoding strategy.

### API Reference

**`ESM3SampleInput`**
| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Protein sequence(s) to mutate. Accepts a single string or list of strings |

**`ESM3SampleConfig`**
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_checkpoint` | `str` | `"esm3_sm_open_v1"` | ESM3 model checkpoint to use |
| `temperature` | `float` | `1.0` | Sampling temperature for amino acid selection |
| `decoding_method` | `str` | `"entropy"` | Method for selecting positions to mutate: `"entropy"`, `"max_logit"`, `"random"` |
| `num_mutations` | `int` | `1` | Number of positions to mutate per sequence |
| `batch_size` | `Optional[int]` | `None` | Number of sequences to process per batch |
| `return_logits` | `bool` | `False` | Whether to include per-position logits in the output |

**`ESM3SampleOutput`**
| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Sampled/mutated protein sequences |
| `logits` | `Optional[List[List[List[float]]]]` | Per-position amino acid logits (num_sequences, seq_len, 20) |

In [3]:
inputs = ESM3SampleInput(sequences=["MVLSPADKTNVKAAW"])

# Configure mutation sampling
config = ESM3SampleConfig(
    decoding_method="entropy",
    num_mutations=5,
)

sample_result = run_esm3_sample(inputs, config)
print(f"Original: {inputs.sequences[0]}")
print(f"Mutated:  {sample_result.sequences[0]}")


Original: MVLSPADKTNVKAAW
Mutated:  MVFRPADEANVKADW


## 📏 3. Score Sequences

Compute pseudo-perplexity metrics for each sequence. Pseudo-perplexity measures how "natural" a protein sequence looks to the model. Lower values indicate sequences that are more consistent with the patterns learned from natural proteins. This is useful for ranking designed sequences or identifying anomalous regions.

### API Reference

**`ESM3ScoringInput`**
| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Protein sequence(s) to score. Accepts a single string or list of strings |

**`ESM3ScoringConfig`**
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_checkpoint` | `str` | `"esm3_sm_open_v1"` | ESM3 model checkpoint to use |
| `batch_size` | `int` | `32` | Number of masked sequences to process per forward pass |
| `return_logits` | `bool` | `False` | Whether to include per-position logits in the output |

**`ESM3ScoringOutput`**
| Field | Type | Description |
|-------|------|-------------|
| `scores` | `List[SequenceScores]` | List of scoring outputs, one per input sequence |

In [4]:
inputs = ESM3ScoringInput(sequences=["MKTAYIAKQR", "EVQLVESGGS"])

# Configure scoring parameters
config = ESM3ScoringConfig(batch_size=16, return_logits=False)

score_result = run_esm3_score(inputs, config)
print(f"Sequence 1: {inputs.sequences[0]}")
print(f"    Perplexity: {score_result.scores[0].metrics['perplexity']:.3f}")
print(f"Sequence 2: {inputs.sequences[1]}")
print(f"    Perplexity: {score_result.scores[1].metrics['perplexity']:.3f}")


Sequence 1: MKTAYIAKQR
    Perplexity: 16.108
Sequence 2: EVQLVESGGS
    Perplexity: 16.014


## 🧱 4. Predict Structure

Generate a 3D structure for a protein sequence. ESM3 can predict 3D protein structures from amino acid sequences, providing per-residue confidence scores (pLDDT). Higher pLDDT values indicate more reliable predictions.

### API Reference

**`ESM3StructurePredictionInput`**
| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Protein sequence(s) to predict structure for. Accepts a single string or list of strings |

**`ESM3StructurePredictionConfig`**
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_checkpoint` | `str` | `"esm3_sm_open_v1"` | ESM3 model checkpoint to use |
| `batch_size` | `int` | `128` | Number of sequences to process in parallel |

**`ESM3StructurePredictionOutput`**
| Field | Type | Description |
|-------|------|-------------|
| `structures` | `List[Dict[str, Any]]` | Predicted structures with `sequence`, `pdb_string`, `avg_plddt`, and `ptm` |
| `num_sequences` | `int` | Number of sequences processed |

In [5]:
inputs = ESM3StructurePredictionInput(sequences=["MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG"])

# Configure structure prediction
config = ESM3StructurePredictionConfig(batch_size=1)

structure_result = run_esm3_structure_prediction(inputs, config)
print(f"Avg pLDDT: {structure_result.structures[0]['avg_plddt']:.2f}")


Avg pLDDT: 0.91


## 💾 5. Export Results

Save the results to files for downstream analysis.

### Supported formats:
- **JSON** -- Structured data with metadata and all scoring information
- **CSV** -- Tabular format for embeddings, compatible with pandas and other data tools
- **FASTA** -- Simple sequence format for compatibility with other bioinformatics tools
- **PDB** -- Standard 3D structure format for molecular visualization and analysis

The exported results can be used for:
- Downstream machine learning (embeddings as features)
- Sequence analysis and comparison (sampled variants)
- Ranking and filtering protein designs (scoring metrics)
- Molecular visualization and structural analysis (predicted structures)

In [ ]:
# Export embeddings to CSV format
embeddings_result.export("./example_output/esm3_embeddings", file_format="csv")
print("Embeddings exported to ./example_output/esm3_embeddings")

# Export sampled sequences to FASTA format
sample_result.export("./example_output/esm3_sequences", file_format="fasta")
print("Sampled sequences exported to ./example_output/esm3_sequences")

# Export scores to JSON format
score_result.export("./example_output/esm3_scores", file_format="json")
print("Scores exported to ./example_output/esm3_scores")

# Export predicted structure to PDB format
structure_result.export("./example_output/esm3_structure", file_format="pdb")
print("Structure exported to ./example_output/esm3_structure")